# Whole-Life Carbon & Life Cycle Costing Assessment (EN 15978 / RICS 2nd Edition)

**Lead Engineer**: Fraser James Mclellan  
**Standards**: EN 15978:2011, BS ISO 15686-5:2017, RICS Professional Standard: Whole life carbon assessment for the built environment (2nd Edition, 2023)  
**Archetype**: 5-Storey Commercial Office Building, $7,500\text{ m}^2\text{ GIA}$ ($1,500\text{ m}^2$ floorplate, $160\text{ m}$ perimeter, $45\%$ WWR, Edinburgh context)  

### Core Engineering Research Question
> *Under a rapidly decarbonising electrical grid, does an ultra-efficient, low-carbon new build ever overcome the upfront demolition and construction carbon deficit compared to a retained-structure deep retrofit?*

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import BuildingArchetype, EconomicParameters, ScenarioConfig, SCENARIO_CONFIGS
from src.emission_factors import EmissionFactorDatabase
from src.grid_decarbonisation import GridDecarbonisationModel
from src.inventory import BuildingInventory
from src.carbon_calculator import WholeLifeCarbonEngine
from src.lcc_calculator import LifeCycleCostingEngine
from src.benchmarking import SustainabilityBenchmarkScorer
from src.plotting import WLCPlotter

print("Engine modules loaded successfully.")

## 1. Archetype Building Geometry & Scenario Definitions

We model 4 distinct intervention scenarios:
1. **Scenario 1 — Baseline Counterfactual**: Existing 1980s concrete framed building retained; like-for-like condensing gas boiler replacement, standard air-cooled chiller (R410A), uninsulated facade ($140\text{ kWh/m}^2/\text{yr}$ EUI).
2. **Scenario 2 — Deep Retrofit**: 100% substructure & superstructure retained; high-performance triple glazing, bio-insulation, central ASHP with **R290 propane** (GWP = 3), MVHR, rooftop PV ($55\text{ kWh/m}^2/\text{yr}$ EUI).
3. **Scenario 3 — Standard New Build**: Demolition of existing building; structural steel frame, composite slabs, standard ASHP with **R32** (GWP = 675), compliant envelope ($55\text{ kWh/m}^2/\text{yr}$ EUI).
4. **Scenario 4 — Exemplar Timber New Build**: Selective deconstruction; 70% GGBS concrete foundations, Glulam/CLT timber superstructure, Passivhaus envelope, R290 ASHP, BIPV ($45\text{ kWh/m}^2/\text{yr}$ EUI).

In [ ]:
sc_rows = []
for sc_id, sc in SCENARIO_CONFIGS.items():
    sc_rows.append({
        "Scenario": sc.short_name,
        "Structure Retained": f"{int(sc.structure_retained_pct * 100)}%",
        "U-Wall [W/m²K]": sc.u_wall,
        "U-Window [W/m²K]": sc.u_window,
        "q50 [m³/(h·m²)]": sc.q50_air_permeability,
        "Heating COP": sc.heating_cop,
        "Refrigerant": sc.refrigerant_type,
        "Net EUI [kWh/m²/yr]": sc.net_operational_eui,
        "Upfront Capex [£/m²]": f"£{sc.upfront_capex_per_m2:.0f}",
    })
pd.DataFrame(sc_rows)

## 2. Dynamic UK Electrical Grid Decarbonisation (2026–2086)

Under **EN 15978** and the **RICS 2nd Edition**, operational carbon over the 60-year Reference Study Period must account for power sector decarbonisation.
We evaluate three trajectories:
- **DESNZ Green Book Central**: HM Treasury appraisal baseline.
- **National Grid ESO FES 'Leading the Way'**: Rapid decarbonisation ($<0.005\text{ kg CO}_2\text{e/kWh}$ by 2045).
- **Static SAP 10.2**: Constant $0.136\text{ kg CO}_2\text{e/kWh}$ counterfactual.

In [ ]:
grid = GridDecarbonisationModel()
fig, ax = plt.subplots(figsize=(9, 4.5))
years = grid.years
ax.plot(years, grid.get_trajectory_series("desnz_central"), label="DESNZ Green Book Central", color="#1f77b4", linewidth=2)
ax.plot(years, grid.get_trajectory_series("fes_leading_way"), label="National Grid ESO 'Leading the Way'", color="#2ca02c", linewidth=2)
ax.plot(years, grid.get_trajectory_series("sap10_static"), label="Static SAP 10.2 Counterfactual", color="#d62728", linestyle="--", linewidth=1.5)
ax.set_xlabel("Calendar Year", color="#000000", fontweight="bold")
ax.set_ylabel("Grid Emission Factor [kg CO₂e/kWh]", color="#000000", fontweight="bold")
ax.set_title("Dynamic UK Electricity Grid Decarbonisation Trajectories (2026-2086)", color="#000000", fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

## 3. CIBSE TM65 MEP Embodied Carbon & Refrigerant Leakage

Under **CIBSE TM65 §4.2** and **Regulation (EU) 2024/573 (F-Gas)**, heat pump refrigerant fugitive leakage represents a massive hidden emissions driver.
We calculate peak thermal heating/cooling loads and determine 60-year refrigerant leakage and end-of-life venting.

In [ ]:
inv = BuildingInventory()
db = EmissionFactorDatabase()
ref_data = []
for sc_id, sc in SCENARIO_CONFIGS.items():
    thermal = inv.calculate_peak_thermal_load(sc)
    charge = thermal["initial_refrigerant_charge_kg"]
    gwp = db.get_refrigerant_gwp(sc.refrigerant_type)
    annual_leak_kg = charge * sc.annual_leakage_rate * gwp
    life_leak_tonnes = (annual_leak_kg * 60.0) / 1000.0
    ref_data.append({
        "Scenario": sc.short_name,
        "Peak Load [kWth]": round(thermal["installed_capacity_kw"], 1),
        "Refrigerant": sc.refrigerant_type,
        "GWP": gwp,
        "Initial Charge [kg]": round(charge, 1),
        "60-Yr Fugitive Leakage [t CO₂e]": round(life_leak_tonnes, 2),
        "Intensity [kg CO₂e/m² GIA]": round((life_leak_tonnes * 1000.0) / 7500.0, 1),
    })
pd.DataFrame(ref_data)

## 4. Whole-Life Carbon Accounting & Life Cycle Module Breakdown

Now we run the complete **EN 15978 / RICS 2nd Edition** carbon accounting engine across Modules A1–C4 and separate Module D circular credits.

In [ ]:
carbon_engine = WholeLifeCarbonEngine(db=db, grid=grid, inventory=inv)
carbon_results = carbon_engine.calculate_all()

plotter = WLCPlotter()
plotter.plot_fig1_wlc_stage_breakdown(carbon_results)
plotter.plot_fig3_spatial_elemental_breakdown(carbon_results)
plotter.plot_fig4_mep_refrigerant_fgas_tradeoff(carbon_results)

summary_table = []
for sc_id, res in carbon_results.items():
    summary_table.append({
        "Scenario": SCENARIO_CONFIGS[sc_id].short_name,
        "A1-A5 Upfront [kg/m²]": round(res.upfront_per_m2(), 1),
        "B1-B5 In-Use [kg/m²]": round(res.b1_b5_inuse_embodied / res.gia, 1),
        "B6 Operational [kg/m²]": round(res.operational_per_m2("desnz_central"), 1),
        "C1-C4 End of Life [kg/m²]": round(res.c1_c4_end_of_life / res.gia, 1),
        "Net WLC (A1-C4) [kg/m²]": round(res.total_wlc_per_m2("desnz_central"), 1),
        "Module D Credits [kg/m²]": round(res.module_d_credits / res.gia, 1),
    })
pd.DataFrame(summary_table)

## 5. Dynamic 60-Year Cumulative Carbon Payback & Tipping-Point Dynamics

We solve for the breakeven year $T^*$:
$$\Delta E_{\text{upfront}} + \sum_{t=1}^{T^*} \Delta E_{\text{in-use embodied}, t} \le \sum_{t=1}^{T^*} \left( E_{\text{oper}, B}(t) - E_{\text{oper}, A}(t) \right)$$

In [ ]:
plotter.plot_fig2_cumulative_carbon_payback(carbon_results)

payback_retro = carbon_engine.calculate_payback_years(
    carbon_results["scenario_1"], carbon_results["scenario_2"], "desnz_central"
)
payback_new = carbon_engine.calculate_payback_years(
    carbon_results["scenario_2"], carbon_results["scenario_3"], "desnz_central"
)
payback_fes = carbon_engine.calculate_payback_years(
    carbon_results["scenario_2"], carbon_results["scenario_3"], "fes_leading_way"
)

print(f"Deep Retrofit payback vs Gas Baseline: Year {payback_retro} (DESNZ Central)")
print(f"Standard New Build vs Deep Retrofit breakeven: {'Never (Irrecoverable Deficit)' if payback_new is None else f'Year {payback_new}'}")
print(f"Under ESO Rapid Decarb breakeven: {'Never (Irrecoverable Deficit)' if payback_fes is None else f'Year {payback_fes}'}")

## 6. Life Cycle Costing (BS ISO 15686-5) & Techno-Economic Pareto Frontier

We compute the **Net Present Value (NPV)** over 60 years using **HM Treasury Green Book** discount rates ($3.5\% \to 3.0\%$) and determine the **Marginal Abatement Cost (MAC)**.

In [ ]:
lcc_engine = LifeCycleCostingEngine(econ_params=EconomicParameters(), archetype=BuildingArchetype(), grid_model=grid)
lcc_results = lcc_engine.calculate_all(carbon_results)

plotter.plot_fig5_techno_economic_carbon_cost_frontier(carbon_results, lcc_results)

lcc_table = []
for sc_id, l_res in lcc_results.items():
    lcc_table.append({
        "Scenario": SCENARIO_CONFIGS[sc_id].short_name,
        "Upfront Capex [£/m²]": round(l_res.upfront_capex_per_m2(), 0),
        "Maint Opex NPV [£/m²]": round(l_res.discounted_maintenance_opex / l_res.gia, 0),
        "Energy Bills NPV [£/m²]": round(l_res.discounted_energy_bills / l_res.gia, 0),
        "Replacements NPV [£/m²]": round(l_res.discounted_replacement_capex / l_res.gia, 0),
        "Total Financial NPV [£/m²]": round(l_res.financial_npv_per_m2(), 0),
        "Social Carbon NPV [£/m²]": round(l_res.discounted_societal_carbon_cost / l_res.gia, 0),
        "MAC [£/t CO₂e abated]": round(l_res.marginal_abatement_cost_gbp_per_tonne, 1) if sc_id != "scenario_1" else "-",
    })
pd.DataFrame(lcc_table)

## 7. Sustainability Benchmarks (LETI, RIBA 2030, UK NZCBS)

We score each scenario against UK Net Zero targets.

In [ ]:
scorer = SustainabilityBenchmarkScorer()
benchmarks = scorer.score_all(carbon_results)
plotter.plot_fig6_leti_riba_benchmark_compliance(carbon_results, benchmarks)

score_table = []
for sc_id, b in benchmarks.items():
    score_table.append({
        "Scenario": SCENARIO_CONFIGS[sc_id].short_name,
        "Upfront [kg/m²]": round(b.upfront_kg_per_m2, 1),
        "LETI Band": b.leti_embodied_band,
        "RIBA 2025": "PASS" if b.riba_2025_pass else "FAIL",
        "RIBA 2030": "PASS" if b.riba_2030_pass else "FAIL",
        "NZCBS Upfront": "PASS" if b.nzcbs_upfront_pass else "FAIL",
        "NZCBS EUI": "PASS" if b.nzcbs_operational_pass else "FAIL",
    })
pd.DataFrame(score_table)